In [0]:
import requests
import json
from datetime import date, timezone, timedelta

In [0]:
%run ./config_notebook

In [0]:
dbutils.widgets.text("load_date", "")

today = dbutils.widgets.get("load_date")

if today:
    today=date.fromisoformat(today)
else:
    today=date.today()
    
start_date = today - timedelta(days=3)

start_date = str(start_date)
end_date = str(today)

In [0]:
params = [
    ("_lastUpdated", "ge" + start_date),
    ("_lastUpdated", "lt" + end_date),
    ("_count", "100")
]

In [0]:
# Extracts paginated FHIR resources from the API and saves each response as raw JSON files.
for resource_name,resource_url in base_urls.items():
    print("Starting:",resource_name)
    next_url=resource_url
    page=1
    try:
        while next_url:
            if page==1:
                response=requests.get(next_url,params=params,headers={"Accept":"application/fhir+json"},timeout=60)
            else:
                response=requests.get(next_url,headers={"Accept":"application/fhir+json"},timeout=60)
            response.raise_for_status()
            data=response.json()
            folder=f"{raw_root}/{resource_name.lower()}/extraction_date={end_date}"
            dbutils.fs.mkdirs(folder)
            file_path=f"{folder}/{resource_name.lower()}_page_{page}.json"
            dbutils.fs.put(file_path,response.text,True)
            next_url=None
            for link in data.get("link",[]):
                if link.get("relation")=="next":
                    next_url=link.get("url")
                    break
            page=page+1
    except Exception as error:
        print(resource_name,"failed:",error)

In [0]:
dbutils.notebook.exit("success")